# IFEval: the protocol grid — solo and panel, with and without correction

IFEval contains 541 instruction-following prompts with deterministic checks for requirements such
as word counts, required sections, and forbidden punctuation. Grading uses the vendored official
verifier and makes no grading-model calls.

This notebook runs the SAME benchmark (`ifeval`) across a 2x2 protocol grid, varying exactly one
dimension at a time:

|                | solo                    | panel                                     |
|----------------|-------------------------|-------------------------------------------|
| **no loop**    | plain `sf.Model`        | `sf.Fusion` (drafts blended once)         |
| **corrective** | `sf.SelfCorrective`     | `sf.CorrectiveLoop` (drafts checked, best |
|                | (self-coached retries)  | passing draft submitted verbatim)         |

The corrective recipes work on ANY benchmark that advertises a check surface in its manifest —
IFEval's is free (deterministic verifier), so a loop here spends only on members and the judge.
Members draft, the check marks each draft, the first passing draft is submitted word-for-word,
and a no-pass round buys one coaching call plus a retry, up to `max_rounds` (a cost cap, not a
target).

Every evaluation cell below performs paid Candidate calls. Start with `limit=1`, inspect the
Reports, and increase the selection deliberately.

## Before running

From a terminal in `packages/screamingface/`:

```bash
just stack-prepare  # first run only: download pinned Benchmark assets
just stack-up       # start AI Gateway :9105 and Engine :9108
just stack-status
```

Use `just stack-logs` to inspect startup failures and `just stack-down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

In [ ]:
import screamingface as sf

sf.connect()

## Define Candidate-owned answer and synthesis policy

An explicit synthesizer `sf.Model` makes its whole-Fusion prompt and generation parameters
visible. The corrective recipes reuse the same routes: the judge model tie-breaks between
passing drafts and coaches failed rounds under the loop's own revisioned instructions.

In [ ]:
ANSWER_PROMPT = (
    "Answer the request accurately and completely. "
    "Follow every instruction and formatting constraint in the request."
)
SYNTHESIS_PROMPT = (
    "Produce one final answer to the original request from the panel drafts. "
    "Preserve every instruction and formatting constraint."
)

haiku = sf.Model(
    "openrouter/anthropic/claude-haiku-4.5",
    prompt=ANSWER_PROMPT,
    params={"max_tokens": 4096},
)
gemini = sf.Model(
    "openrouter/google/gemini-3-flash-preview",
    prompt=ANSWER_PROMPT,
    params={"max_tokens": 4096},
)
kimi = sf.Model(
    "openrouter/moonshotai/kimi-k2.6",
    prompt=ANSWER_PROMPT,
    params={"max_tokens": 4096},
)
panel = sf.Fusion(
    [haiku, gemini],
    name="haiku+gemini",
    synthesizer=sf.Model(
        "openrouter/moonshotai/kimi-k2.6",
        prompt=SYNTHESIS_PROMPT,
        params={"max_tokens": 4096},
    ),
)
[kimi, panel]

## 1. Solo, no loop — canonical baseline

In [ ]:
canonical_solo = sf.evaluate(kimi, benchmark="ifeval", limit=1)
canonical_solo

## 2. Panel, no loop — whole-Fusion synthesis

In [ ]:
canonical_fusion = sf.evaluate(panel, benchmark="ifeval", limit=1)
canonical_fusion

## 3. Solo, corrective — `sf.SelfCorrective`

The same model re-sits the exam up to three times, authoring its own study notes from the
check surface's sanitized feedback between sittings. A first-round pass costs one draft and
one free check — nothing else.

In [ ]:
self_corrective = sf.evaluate(
    sf.SelfCorrective(kimi, max_rounds=3),
    benchmark="ifeval",
    limit=1,
)
self_corrective

## 4. Panel, corrective — `sf.CorrectiveLoop`

Both members draft in parallel; the first passing draft is submitted verbatim (the judge only
tie-breaks between multiple passers and coaches a no-pass round). Switching this loop to another
check-capable benchmark is one changed `benchmark=` line — the loop itself is
benchmark-independent.

In [ ]:
corrective_loop = sf.evaluate(
    sf.CorrectiveLoop([haiku, gemini], judge=kimi, max_rounds=3),
    benchmark="ifeval",
    limit=1,
)
corrective_loop

## Compare the grid

Four complete portable artifacts — same benchmark, same models, one protocol dimension varied
at a time. Read score against cost: the corrective column's spend scales with how many rounds
each case actually bought.

In [ ]:
{
    name: {"score": report.candidates[0].score, "usage": report.usage.to_dict()}
    for name, report in {
        "canonical_solo": canonical_solo,
        "canonical_fusion": canonical_fusion,
        "self_corrective": self_corrective,
        "corrective_loop": corrective_loop,
    }.items()
}

In [ ]:
{
    "canonical_solo": canonical_solo.to_dict(),
    "canonical_fusion": canonical_fusion.to_dict(),
    "self_corrective": self_corrective.to_dict(),
    "corrective_loop": corrective_loop.to_dict(),
}